# Taxi Trip Cleanup

In this notebook we clean the taxi trip data provided by the city of chicago: https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data (downloaded on 05.05.2026).

In [2]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd

# import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np
from scipy.stats import zscore

import geopandas as gpd

import fastparquet

ModuleNotFoundError: No module named 'geopandas'

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load data                               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

raw_data= pd.read_csv("../data/tripdata/Taxi_Trips_raw.csv")

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Data overview.                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

print(f"Raw data shape: {raw_data.shape}")
print(f"Raw data info:")
print(raw_data.info())
print(f"Raw data columns: {list(raw_data.columns)}")
raw_data.head()

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Data type referencing                   #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# turn strings to numeric values
stringToNumeric = ["Trip Miles", "Fare", "Tips", "Tolls", "Extras", "Trip Total"]
raw_data[stringToNumeric] = raw_data[stringToNumeric].apply(
    lambda s: pd.to_numeric(
        s.astype(str)
         .str.replace("$", "", regex=False)
         .str.replace(",", ".", regex=False),
        errors="coerce"
    )
)


# Change type of Trip_Start_Timestamp and Trip_End_Timestamp to datetime object
raw_data["Trip Start Timestamp"] = pd.to_datetime(
    raw_data["Trip Start Timestamp"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce"
)

raw_data["Trip End Timestamp"] = pd.to_datetime(
    raw_data["Trip End Timestamp"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce"
)

raw_data.info()

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Select Time Range                       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #
# check range of timestamps
print("Start Timestamp range:", raw_data["Trip Start Timestamp"].min(), "to", raw_data["Trip Start Timestamp"].max())

# remove 2024 data so we have 1 year (2025) data
raw_data = raw_data[
    raw_data["Trip Start Timestamp"].between(
        pd.Timestamp("2025-01-01"), pd.Timestamp("2026-01-01 00:00:00")
    )
]
print("After filtering to 2025, Start Timestamp range:", raw_data["Trip Start Timestamp"].min(), "to", raw_data["Trip Start Timestamp"].max())

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Remove Duplicates                       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# Check for duplicates
n_duplicates = raw_data.duplicated().sum()
print(f"Number of duplicate rows: {n_duplicates}")

# Check if any trip ids are duplicate
n_duplicate_trip_ids = raw_data["Trip ID"].duplicated().sum()
print(f"Number of duplicated Trip IDs: {n_duplicate_trip_ids}")

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Remove N/A-values                       #
#  (only non-location data)               #
# # # # # # # # # # # # # # # # # # # # # #

# Check for NA values
na_summary = (
    raw_data
    .isna()
    .sum()
    .to_frame(name="Null Count")
    .assign(
        Null_Percent=lambda x: (x["Null Count"] / len(raw_data) * 100).round(2)
    )
)
na_summary

In [ ]:
# Set the columns to check for NA values (e.g., exclude location columns)
columns_to_remove = ['Taxi ID', 'Trip End Timestamp', 'Trip Seconds', 'Trip Miles', "Fare", "Tips", "Tolls", "Extras", "Trip Total"]  # change as needed

# Calculate how many rows would be removed if NA in these columns only
removed_na = raw_data.shape[0] - raw_data.dropna(subset=columns_to_remove).shape[0]
removed_rel_share = (removed_na / raw_data.shape[0]) * 100

print(f'Dropping rows with NA in {columns_to_remove} would remove {removed_na} rows ({removed_rel_share:.2f}%) of the dataset.')


In [ ]:
# Remove na values, see upper cell
raw_data = raw_data.dropna(subset=columns_to_remove)

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Trip Data Cleaning                      #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# Trip miles < 0
n_negative_miles = (raw_data["Trip Miles"] <= 0).sum()
print(f"Number of negative trip miles: {n_negative_miles}")

# Trip seconds < 60 (s)
n_seconds_less_min = (raw_data["Trip Seconds"] < 60).sum()
print(f"Number of trips faster than 1 min: {n_seconds_less_min}")

# Remove negative trip miles and trips faster than 60s
raw_data = raw_data[((raw_data["Trip Miles"] > 0) & (raw_data["Trip Seconds"] >= 60))]

overlapping_trips = raw_data.groupby('Taxi ID').apply(
    lambda group: group[
        (group['Trip Start Timestamp'].lt(group['Trip End Timestamp'].shift(-1))) &
        (group['Trip End Timestamp'].gt(group['Trip Start Timestamp'].shift(-1)))
    ]
)
print("Number of Taxi IDs with overlapping operation time periods:", len(overlapping_trips))
overlapping_trips

# assumption: if there are overlapping trips with the same taxi id, there was something wrong with all of these trips, so therefore we remove all of the overlapping trips
overlapping_trips_indices = overlapping_trips.index.get_level_values(1)
raw_data = raw_data.drop(overlapping_trips_indices, axis= 0)


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Price Data                              #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# check for prices <= 0
# since tips, tolls and extras are dependent on the details of the trips and might be 0, it is okay for them to be 0.
# Fares are mandatory and have to be greater than 0

fare_check = (raw_data['Fare'] <= 0).sum()
tips_check = (raw_data['Tips'] < 0).sum()
tolls_check = (raw_data['Tolls'] < 0).sum()
extras_check = (raw_data['Extras'] < 0).sum()
print("Number of entries with fares <= 0:", fare_check)
print("Number of entries with tips <= 0:", tips_check)
print("Number of entries with tolls <= 0:", tolls_check)
print("Number of entries with extras <= 0:", extras_check)

# remove negative or 0 fares
raw_data = raw_data[raw_data["Fare"] > 0]
raw_data.reset_index(drop=True)

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Outlier Detection                       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

numeric_columns = ["Trip Seconds", "Trip Miles", "Fare", "Tips", "Tolls", "Extras", "Trip Total"]

# 2 plots per row
rows = math.ceil(len(numeric_columns) / 2)

fig, axes = plt.subplots(rows, 2, figsize=(12, rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_columns):
    data = pd.to_numeric(raw_data[col], errors='coerce').dropna()
    axes[i].boxplot(data, vert=True)
    axes[i].set_title(f'Boxplot of {col}')
    axes[i].set_xlabel(col)
    axes[i].grid(True)

# Remove any unused axes
for j in range(len(numeric_columns), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
zscore_threshold = 2
outlier_indices_per_column = {}
all_outlier_indices = set()

# remove outliers based on z-score
for col in numeric_columns:
    # Replace zeros with NaN and filter out non-positive values
    safe_col = raw_data[col].replace(0, np.nan)
    safe_col = safe_col[safe_col > 0]
    
    if safe_col.empty:
        continue
    
    # Log-transform and compute z-score
    log_transformed = np.log(safe_col)
    zscores = np.abs(zscore(log_transformed))
    zscore_series = pd.Series(zscores, index=log_transformed.index)

    # Find outliers
    outlier_indices = zscore_series[zscore_series > zscore_threshold].index
    outlier_indices_per_column[col] = list(outlier_indices)
    all_outlier_indices.update(outlier_indices)

    print(f"--> {len(outlier_indices)} outliers detected in '{col}'")

# Drop all outliers at once
total_rows = len(raw_data) # for dropped percentage of dataset
df_cleaned = raw_data.drop(index=all_outlier_indices).reset_index(drop=True)

removed_rows = len(all_outlier_indices)
removed_percentage = (removed_rows / total_rows) * 100

print(f"\nOutlier detection completed. Total rows removed: {len(all_outlier_indices)}, {removed_percentage}")

In [ ]:
import pandas as pd
import geopandas as gpd

# --- 1) Fix comma-decimal coordinates ---
coord_cols = [
    "Pickup Centroid Latitude", "Pickup Centroid Longitude",
    "Dropoff Centroid Latitude", "Dropoff Centroid Longitude"
]

raw_data[coord_cols] = raw_data[coord_cols].apply(
    lambda s: pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False),
                            errors="coerce")
)

# --- 2) Load tract polygons ---
tracts = gpd.read_file("../data/tripdata/Census_Tracts_2024.geojson").to_crs(4326)
TRACT_COL = "census_t_1"

centroids = tracts.set_index(TRACT_COL).geometry.centroid

# --- 3) Helper: fill missing tracts from coords ---
def fill_tract(lat_col, lon_col, tract_col):
    mask = raw_data[tract_col].isna() & raw_data[lat_col].notna()
    if mask.any():
        pts = gpd.GeoDataFrame(
            geometry=gpd.points_from_xy(raw_data.loc[mask, lon_col],
                                        raw_data.loc[mask, lat_col]),
            crs=4326,
            index=raw_data.index[mask]
        )
        joined = gpd.sjoin(pts, tracts[[TRACT_COL, "geometry"]],
                           how="left", predicate="within")
        raw_data.loc[joined.index, tract_col] = joined[TRACT_COL].values

# --- 4) Helper: fill missing coords from tract ---
def fill_coords(lat_col, lon_col, tract_col):
    mask = raw_data[tract_col].notna() & raw_data[lat_col].isna()
    if mask.any():
        c = centroids.reindex(raw_data.loc[mask, tract_col].astype(str))
        raw_data.loc[mask, lat_col] = c.y.values
        raw_data.loc[mask, lon_col] = c.x.values

# --- 5) Apply ---
raw_data["Pickup Census Tract"] = raw_data["Pickup Census Tract"].astype("string")
raw_data["Dropoff Census Tract"] = raw_data["Dropoff Census Tract"].astype("string")

fill_tract("Pickup Centroid Latitude", "Pickup Centroid Longitude", "Pickup Census Tract")
fill_tract("Dropoff Centroid Latitude", "Dropoff Centroid Longitude", "Dropoff Census Tract")

fill_coords("Pickup Centroid Latitude", "Pickup Centroid Longitude", "Pickup Census Tract")
fill_coords("Dropoff Centroid Latitude", "Dropoff Centroid Longitude", "Dropoff Census Tract")

print(raw_data.isna().sum())


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Remove N/A-values                       #
#  (only non-location data)               #
# # # # # # # # # # # # # # # # # # # # # #

# Check for NA values
na_summary = (
    raw_data
    .isna()
    .sum()
    .to_frame(name="Null Count")
    .assign(
        Null_Percent=lambda x: (x["Null Count"] / len(raw_data) * 100).round(2)
    )
)
na_summary

In [ ]:
# Set the columns to check for NA values (e.g., exclude location columns)
columns_to_remove = ["Pickup Census Tract", "Dropoff Census Tract", "Pickup Community Area", "Dropoff Community Area", "Pickup Centroid Latitude", "Pickup Centroid Longitude", "Pickup Centroid Location", "Dropoff Centroid Latitude", "Dropoff Centroid Longitude", "Dropoff Centroid  Location"]

# Calculate how many rows would be removed if NA in these columns only
removed_na = raw_data.shape[0] - raw_data.dropna(subset=columns_to_remove).shape[0]
removed_rel_share = (removed_na / raw_data.shape[0]) * 100

print(f'Dropping rows with NA in {columns_to_remove} would remove {removed_na} rows ({removed_rel_share:.2f}%) of the dataset.')


### Removing all N/A values from the dataset would remove another 4.80% of the dataset. This is acceptable, therefore we throw out all the N/A values

In [ ]:
# Remove na values, see upper cell
raw_data = raw_data.dropna(subset=columns_to_remove)

In [ ]:
raw_data.head()

In [ ]:
# write cleaned dataframe into .parquet file
raw_data.to_parquet("../data/tripdata/raw_data.parquet",index=None)

In [ ]:
raw_data.info()